In [4]:
from transformers import AutoProcessor, AutoConfig
import transformers

checkpoint = "openai/whisper-small"

#加载processor与模型配置，但不下载模型权重
processor = AutoProcessor.from_pretrained(checkpoint)
whisper_config = AutoConfig.from_pretrained(checkpoint)

feature_extractor = processor.feature_extractor

print("Transformers版本:", transformers.__version__)
print("Checkpoint:", checkpoint)
print("Processor类型:", type(processor).__name__)
print("FeatureExtractor类型:", type(feature_extractor).__name__)
print("Tokenizer类型:", type(processor.tokenizer).__name__)

print("\n--- Feature Extractor配置 ---")

fields = [
    "sampling_rate",
    "feature_size",
    "n_fft",
    "hop_length",
    "chunk_length",
    "n_samples",
    "nb_max_frames",
    "padding_side",
    "padding_value",
    "return_attention_mask",
]

for field in fields:
    print(f"{field}:{getattr(feature_extractor, field, '未提供')}")

print("\n--- 模型配置 ---")
print("num_mel_bins:", whisper_config.num_mel_bins)
print("max_source_positions:", whisper_config.max_source_positions)

# 验证关键参数的关系
assert feature_extractor.sampling_rate == 16000
assert feature_extractor.feature_size == 80
assert feature_extractor.n_samples == 480000
assert feature_extractor.nb_max_frames == 3000

assert (
    feature_extractor.feature_size
    == whisper_config.num_mel_bins
)

print("\nwhisper-small配置检查通过")

Transformers版本: 5.21.0
Checkpoint: openai/whisper-small
Processor类型: WhisperProcessor
FeatureExtractor类型: WhisperFeatureExtractor
Tokenizer类型: WhisperTokenizer

--- Feature Extractor配置 ---
sampling_rate:16000
feature_size:80
n_fft:400
hop_length:160
chunk_length:30
n_samples:480000
nb_max_frames:3000
padding_side:right
padding_value:0.0
return_attention_mask:False

--- 模型配置 ---
num_mel_bins: 80
max_source_positions: 1500

whisper-small配置检查通过


In [5]:
import numpy as np
import pandas as pd
from pathlib import Path

#1.获取Whisper-samll的实际配置
feature_extractor = processor.feature_extractor

sr = feature_extractor.sampling_rate
target_samples = feature_extractor.n_samples
target_frames = feature_extractor.nb_max_frames
feature_size = feature_extractor.feature_size

print("采样率:", sr)
print("标准采样点数:", target_samples)
print("标准帧数:", target_frames)
print("Mel维数:", feature_size)

#2. 生成5s、30s、35s测试波形
#5s和30s使用440Hz正弦波
#35s由前30s的440Hz与后5s的2000Hz组成
def make_tone(duration_sec, frequency, sr):
    t = np.arange(int(duration_sec * sr), dtype=np.float32) / sr
    return  0.1 * np.sin(2 * np.pi * frequency * t)

wave_5s = make_tone(5,440,sr)
wave_30s = make_tone(30,440,sr)
wave_tail = make_tone(5,2000,sr)

wave_35s = np.concatenate([wave_30s, wave_tail])

signals = {
    5: wave_5s,
    30: wave_30s,
    35: wave_35s
}

#3.使用相同的Processor参数
outputs = {}
records = []

for duration, waveform in signals.items():
    result = feature_extractor(
        waveform,
        sampling_rate=sr,
        padding="max_length",
        truncation=True,
        return_tensors="np"
    )

    features = result["input_features"]
    outputs[duration] = features

    original_samples = len(waveform)
    effective_samples = min(original_samples, target_samples)

    padded_samples = max(
        0, target_samples - effective_samples
    )
    truncated_samples = max(
        0, original_samples - target_samples
    )

    records.append({
        "duration_sec": duration,
        "original_samples": original_samples,
        "effective_samples": effective_samples,
        "padded_samples": padded_samples,
        "truncated_samples": truncated_samples,
        "feature_shape": str(tuple(features.shape)),
    })

    print(
        f"{duration}s:",
        "原始采样点数 =", original_samples,
        "输出形状 =", features.shape,
    )

#4. 检查标准输出形状
expected_shape = (1, feature_size, target_frames)

for duration, features in outputs.items():
    assert features.shape == expected_shape
    assert np.isfinite(features).all()

print("\n所有音频的标准输出形状检查通过")

#5.检查35s音频是否丢弃最后5s
#wave_35s的前30s与wave_30s完全一致
assert np.array_equal(
    wave_35s[:target_samples],
    wave_30s
)

max_error = float(
    np.max(np.abs(outputs[35] - outputs[30]))
)

print("35s截断结果与30s输入的最大绝对误差：",max_error)

assert np.allclose(
    outputs[35],
    outputs[30],
    atol=1e-6,
    rtol=1e-6
)

print("35秒截断行为验证通过")

#6.保存processor shape表
df = pd.DataFrame(records)

root = Path.cwd().resolve()
if root.name.lower() == "notebooks":
    root = root.parent

if not (root / "src").is_dir():
    root = Path(r"D:\Projects\speech-ai-learning")

results_dir = root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

output_path = results_dir / "w06d3_processor_shapes.csv"

df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("\nProcessor shape表:")
print(df.to_string(index=False))
print("\n已保存:", output_path)

采样率: 16000
标准采样点数: 480000
标准帧数: 3000
Mel维数: 80
5s: 原始采样点数 = 80000 输出形状 = (1, 80, 3000)
30s: 原始采样点数 = 480000 输出形状 = (1, 80, 3000)
35s: 原始采样点数 = 560000 输出形状 = (1, 80, 3000)

所有音频的标准输出形状检查通过
35s截断结果与30s输入的最大绝对误差： 0.0
35秒截断行为验证通过

Processor shape表:
 duration_sec  original_samples  effective_samples  padded_samples  truncated_samples feature_shape
            5             80000              80000          400000                  0 (1, 80, 3000)
           30            480000             480000               0                  0 (1, 80, 3000)
           35            560000             480000               0              80000 (1, 80, 3000)

已保存: D:\Projects\speech-ai-learning\results\w06d3_processor_shapes.csv


In [6]:
from transformers import WhisperFeatureExtractor, WhisperConfig
from pathlib import Path
import pandas as pd

checkpoints = [
    "openai/whisper-small",
    "openai/whisper-large-v3",
    "openai/whisper-large-v3-turbo"
]

records = []

for checkpoint in checkpoints:
    print(f"\n正在读取:{checkpoint}")

    #只加载特征提取器与配置，不加载模型权重
    extractor = WhisperFeatureExtractor.from_pretrained(checkpoint)
    config = WhisperConfig.from_pretrained(checkpoint)

    record = {
        "checkpoint": checkpoint,
        "sampling_rate": extractor.sampling_rate,
        "feature_size": extractor.feature_size,
        "num_mel_bins": config.num_mel_bins,
        "n_fft": extractor.n_fft,
        "hop_length": extractor.hop_length,
        "chunck_length": extractor.chunk_length,
        "n_samples": extractor.n_samples,
        "nb_max_frames": extractor.nb_max_frames,
        "padding_side": extractor.padding_side,
        "return_attention_mask": extractor.return_attention_mask,
        "max_source_positions": config.max_source_positions,
    }

    records.append(record)

    print("feature_size:",extractor.feature_size)
    print("num_mel_bins:",config.num_mel_bins)
    print("n_fft:", extractor.n_fft)
    print("hop_length:", extractor.hop_length)
    print("chunk_length:", extractor.chunk_length)

    #检查特征维数与模型配置是否一致
    assert extractor.feature_size == config.num_mel_bins

df = pd.DataFrame(records)

print("\nCheckpoint 配置对照表：")
print(df.to_string(index=False))

#保存结果
root = Path.cwd().resolve()

if root.name.lower() == "notebooks":
    root = root.parent

if not (root / "src").is_dir():
    root = Path(r"D:\Projects\speech-ai-learning")

results_dir = root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

output_path = results_dir / "w06d3_checkpoint_comparison.csv"

df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("\n已保存:", output_path)


正在读取:openai/whisper-small
feature_size: 80
num_mel_bins: 80
n_fft: 400
hop_length: 160
chunk_length: 30

正在读取:openai/whisper-large-v3


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

d:\anaconda3\envs\speech-ai\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\fallwind\.cache\huggingface\hub\models--openai--whisper-large-v3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

feature_size: 128
num_mel_bins: 128
n_fft: 400
hop_length: 160
chunk_length: 30

正在读取:openai/whisper-large-v3-turbo


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

d:\anaconda3\envs\speech-ai\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\fallwind\.cache\huggingface\hub\models--openai--whisper-large-v3-turbo. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

feature_size: 128
num_mel_bins: 128
n_fft: 400
hop_length: 160
chunk_length: 30

Checkpoint 配置对照表：
                   checkpoint  sampling_rate  feature_size  num_mel_bins  n_fft  hop_length  chunck_length  n_samples  nb_max_frames padding_side  return_attention_mask  max_source_positions
         openai/whisper-small          16000            80            80    400         160             30     480000           3000        right                  False                  1500
      openai/whisper-large-v3          16000           128           128    400         160             30     480000           3000        right                  False                  1500
openai/whisper-large-v3-turbo          16000           128           128    400         160             30     480000           3000        right                  False                  1500

已保存: D:\Projects\speech-ai-learning\results\w06d3_checkpoint_comparison.csv
